In [31]:
from rag_v2.data_loader import load_all_documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

from rag_v2.embeddings import EmbeddingManager
from rag_v2.store import VectorStore
from rag_v2.indexing import embed_and_store_chunks
from rag_v2.retrieval import RAGRetriever
from rag_v2.answer import rag_answer
from rag_v2.citations import format_sources_generic

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[0]   # notebook/ -> projet
DATA_DIR = PROJECT_ROOT / "data"                 # racine data
docs = load_all_documents(str(DATA_DIR), log_level="INFO")

docs = load_all_documents(str(PROJECT_ROOT / "data" / "pdf_files"), log_level="INFO")

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150, separators=["\n\n", "\n", ". ", " ", ""], length_function=len)
chunks = splitter.split_documents(docs)

embedding_manager = EmbeddingManager()
vectorstore = VectorStore(collection_name="rag_v2_docs", reset=False)
embed_and_store_chunks(chunks, embedding_manager, vectorstore)

retriever = RAGRetriever(vectorstore, embedding_manager)

out = rag_answer("what is backup ?", retriever, mode="personal")
print(out["answer"])
print("\nSOURCE TYPE:", out["source_type"])
if out["source_type"] == "documents":
    print(format_sources_generic(out["citations"]))

100%|██████████| 15/15 [00:00<00:00, 234.07it/s]
2026-03-04 11:31:17,702 | INFO | rag_v2.data_loader | Loaded PDFs: 17
2026-03-04 11:31:17,719 | INFO | rag_v2.data_loader | Loaded .txt   cloud_infrastructure_overview.txt -> 1 docs
2026-03-04 11:31:17,730 | INFO | rag_v2.data_loader | Loaded .txt   cybersecurity_foundations.txt -> 1 docs
2026-03-04 11:31:17,731 | INFO | rag_v2.data_loader | Total loaded documents: 19
100%|██████████| 15/15 [00:00<00:00, 268.37it/s]
2026-03-04 11:31:17,790 | INFO | rag_v2.data_loader | Loaded PDFs: 17
2026-03-04 11:31:17,792 | INFO | rag_v2.data_loader | Total loaded documents: 17
2026-03-04 11:31:18,048 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[EmbeddingManager] Model=text-embedding-3-small | dim=1536
[VectorStore] Collection=rag_v2_docs | count=17
[VectorStore] Upserted 17 chunks | new count=17


2026-03-04 11:31:20,486 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-04 11:31:23,616 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Backup is the process of creating copies of data or system information to ensure its protection and availability in case of data loss, corruption, or system failure. It involves regularly saving data so that it can be restored to a previous state if needed, thereby ensuring data integrity, availability, and compliance with regulatory requirements. Backup procedures typically include monitoring backup jobs, validating successful completion, verifying data integrity, and performing restore tests to confirm that data can be recovered effectively. This is critical for maintaining continuous data protection and supporting disaster recovery efforts.

SOURCE TYPE: documents
- [C1] SOP-IT-0003_Backup_Monitoring.pdf (p.1)
- [C2] SOP-IT-001_Backup_Validation_Procedure.pdf (p.1)
- [C3] SOP-IT-0002_Backup_Restore_Validation.pdf (p.1)
- [C4] SOP-IT-0002_Backup_Restore_Validation.pdf (p.2)
- [C5] WI-IT-0101_Restore_Test_Work_Instruction.pdf (p.1)
- [C6] SOP-IT-005_Disaster_Recovery_Plan.pdf (p.1)
